In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

path = "full_data_6/saft_parameters.csv"


saft = pd.read_csv(path)
raw_cols  = ["m", "sigma", "epsilon_k", "mu", "kappa_ab", "epsilon_k_ab"]
norm_cols = ["m_norm", "sigma_norm", "epsilon_k_norm", "mu_norm", "kappa_ab_norm", "epsilon_k_ab_norm"]
print(f"Substances: {len(saft)}")
saft.head(3)

In [ ]:
# Zero counts and normalization range per parameter
print(f"{'param':15s}  {'zeros':>8s}  {'norm_min':>10s}  {'norm_max':>10s}  {'norm_mean':>10s}  {'norm_std':>10s}")
print("-" * 72)
for rc, nc in zip(raw_cols, norm_cols):
    n_zero = (saft[rc] == 0).sum()
    print(f"{rc:15s}  {n_zero:>5d}/{len(saft):<3d}  "
          f"{saft[nc].min():>10.4f}  {saft[nc].max():>10.4f}  "
          f"{saft[nc].mean():>10.4f}  {saft[nc].std():>10.4f}")

In [ ]:
# Side-by-side histograms: raw vs normalized for all 6 parameters
fig, axes = plt.subplots(6, 2, figsize=(12, 18))
for i, (rc, nc) in enumerate(zip(raw_cols, norm_cols)):
    axes[i, 0].hist(saft[rc], bins=50, edgecolor='k', color='steelblue')
    axes[i, 0].set_title(f"{rc} (raw)")
    axes[i, 0].set_ylabel("count")
    axes[i, 1].hist(saft[nc], bins=50, edgecolor='k', color='darkorange')
    axes[i, 1].set_title(f"{nc} (normalized)")
plt.tight_layout()
plt.show()

In [ ]:
# Non-zero distributions for bimodal parameters
bimodal = ["mu_norm", "kappa_ab_norm", "epsilon_k_ab_norm"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, bimodal):
    nonzero = saft[saft[col] > 0][col]
    zero_frac = (saft[col] == 0).sum() / len(saft)
    ax.hist(nonzero, bins=40, edgecolor='k', color='tomato')
    ax.set_title(f"{col} (non-zero only)\n"
                 f"zeros: {zero_frac:.1%} | non-zero n={len(nonzero)}\n"
                 f"range [{nonzero.min():.3f}, {nonzero.max():.3f}]")
    ax.set_xlabel("normalized value")
    ax.set_ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
# Molecule type breakdown
print("By opt type:")
print(saft["opt"].value_counts())
print("\nBy family (top 15):")
print(saft["family"].value_counts().head(15))

In [ ]:
# Pairplot of normalized params colored by molecule type
types = saft["opt"].astype("category")
type_codes = types.cat.codes
type_labels = types.cat.categories
colors = plt.cm.tab10(np.linspace(0, 1, len(type_labels)))
color_map = {code: colors[code] for code in range(len(type_labels))}

scatter_params = ["m_norm", "sigma_norm", "epsilon_k_norm", "mu_norm", "kappa_ab_norm", "epsilon_k_ab_norm"]
n = len(scatter_params)
fig, axes = plt.subplots(n, n, figsize=(16, 16))

for i, p1 in enumerate(scatter_params):
    for j, p2 in enumerate(scatter_params):
        ax = axes[i, j]
        if i == j:
            for code, label in enumerate(type_labels):
                mask = type_codes == code
                ax.hist(saft.loc[mask, p1], bins=25, alpha=0.5,
                        color=colors[code], label=label)
        else:
            ax.scatter(saft[p2], saft[p1], c=type_codes, cmap='tab10',
                       s=4, alpha=0.4, vmin=0, vmax=len(type_labels)-1)
        if j == 0:
            ax.set_ylabel(p1, fontsize=7)
        if i == n - 1:
            ax.set_xlabel(p2, fontsize=7)
        ax.tick_params(labelsize=6)

# Legend
handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=colors[i],
                       markersize=8, label=label)
           for i, label in enumerate(type_labels)]
fig.legend(handles=handles, loc='upper right', fontsize=9, title='opt type')
plt.suptitle("Normalized parameter pairplot by molecule type", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# log1p normalization for bimodal parameters — compare to current min-max
bimodal_raw = ["mu", "kappa_ab", "epsilon_k_ab"]
bimodal_norm = ["mu_norm", "kappa_ab_norm", "epsilon_k_ab_norm"]

fig, axes = plt.subplots(len(bimodal_raw), 3, figsize=(15, 10))

for i, (rc, nc) in enumerate(zip(bimodal_raw, bimodal_norm)):
    col = saft[rc].values
    # log1p transform, then scale to [0, 1]
    col_log = np.log1p(col)
    col_log_norm = col_log / col_log.max() if col_log.max() > 0 else col_log

    # Current min-max norm
    axes[i, 0].hist(saft[nc], bins=50, edgecolor='k', color='steelblue')
    axes[i, 0].set_title(f"{nc}\n(current min-max)")
    axes[i, 0].set_ylabel("count")

    # log1p normalized
    axes[i, 1].hist(col_log_norm, bins=50, edgecolor='k', color='seagreen')
    axes[i, 1].set_title(f"{rc} log1p-norm\n(proposed)")

    # Non-zero only, log1p normalized
    nz_mask = col > 0
    nz_log = col_log_norm[nz_mask]
    zero_frac = (~nz_mask).sum() / len(col)
    axes[i, 2].hist(nz_log, bins=40, edgecolor='k', color='tomato')
    axes[i, 2].set_title(f"{rc} log1p non-zero only\n"
                         f"zeros: {zero_frac:.1%} | range [{nz_log.min():.3f}, {nz_log.max():.3f}]")

plt.suptitle("log1p normalization vs current min-max (bimodal params)", y=1.01)
plt.tight_layout()
plt.show()

# Stats comparison
print(f"{'param':15s}  {'cur_nonzero_max':>16s}  {'log1p_nonzero_min':>18s}  {'log1p_nonzero_max':>18s}")
print("-" * 75)
for rc, nc in zip(bimodal_raw, bimodal_norm):
    col = saft[rc].values
    col_log_norm = np.log1p(col)
    if col_log_norm.max() > 0:
        col_log_norm = col_log_norm / col_log_norm.max()
    nz = col > 0
    cur_nz_max = saft.loc[nz, nc].max() if nz.any() else 0
    log_nz_min = col_log_norm[nz].min() if nz.any() else 0
    log_nz_max = col_log_norm[nz].max() if nz.any() else 0
    print(f"{rc:15s}  {cur_nz_max:>16.4f}  {log_nz_min:>18.4f}  {log_nz_max:>18.4f}")

In [ ]:
# Log transform of non-zero values — does it help the within-active distribution?
# Compare: linear normalization vs log normalization, non-zero only, for mu and kappa_ab
check_keys = ["mu", "kappa_ab", "epsilon_k_ab"]
fig, axes = plt.subplots(len(check_keys), 2, figsize=(12, 10))

for i, key in enumerate(check_keys):
    col = saft[key].values
    nz = col > 0
    nz_vals = col[nz]

    # Linear: normalize over non-zero range
    lin = (nz_vals - nz_vals.min()) / (nz_vals.max() - nz_vals.min() + 1e-12)

    # Log: log of raw values, then normalize to [0,1]
    log_vals = np.log(nz_vals)
    log_norm = (log_vals - log_vals.min()) / (log_vals.max() - log_vals.min() + 1e-12)

    axes[i, 0].hist(lin, bins=40, edgecolor='k', color='seagreen')
    axes[i, 0].set_title(f"{key} — linear norm (non-zero)\nrange [{nz_vals.min():.4f}, {nz_vals.max():.4f}]")
    axes[i, 0].set_ylabel("count")

    axes[i, 1].hist(log_norm, bins=40, edgecolor='k', color='mediumpurple')
    axes[i, 1].set_title(f"{key} — log norm (non-zero)\nlog range [{log_vals.min():.2f}, {log_vals.max():.2f}]")

plt.suptitle("Linear vs log normalization — non-zero active molecules only", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Proposed: boolean flag + non-zero-only normalization
fig, axes = plt.subplots(len(bimodal_raw), 3, figsize=(15, 10))

for i, rc in enumerate(bimodal_raw):
    col = saft[rc].values
    nz_mask = col > 0

    # Continuous: normalize only over non-zero range → zeros stay 0
    nz_vals = col[nz_mask]
    col_norm_nz = np.zeros_like(col, dtype=float)
    if len(nz_vals) > 0:
        nz_min, nz_max = nz_vals.min(), nz_vals.max()
        col_norm_nz[nz_mask] = (nz_vals - nz_min) / (nz_max - nz_min + 1e-12)

    zero_frac = (~nz_mask).sum() / len(col)

    # Boolean flag bar chart
    axes[i, 0].bar([0, 1], [(~nz_mask).sum(), nz_mask.sum()],
                   color=['steelblue', 'darkorange'], edgecolor='k')
    axes[i, 0].set_xticks([0, 1])
    axes[i, 0].set_xticklabels(['0 (inactive)', '1 (active)'])
    axes[i, 0].set_title(f"{rc}_bool\n{nz_mask.sum()} active ({1-zero_frac:.1%}) / {(~nz_mask).sum()} inactive ({zero_frac:.1%})")
    axes[i, 0].set_ylabel("count")

    # Non-zero normalized continuous (non-zero samples only)
    axes[i, 1].hist(col_norm_nz[nz_mask], bins=40, edgecolor='k', color='seagreen')
    axes[i, 1].set_title(f"{rc}_val — non-zero samples only\nnormalized to [0,1] over non-zero range")

    # Full continuous distribution (zeros + non-zeros together)
    axes[i, 2].hist(col_norm_nz, bins=50, edgecolor='k', color='tomato')
    axes[i, 2].set_title(f"{rc}_val — full distribution\n{(~nz_mask).sum()} exact zeros at 0.0")

plt.suptitle("Proposed: boolean flag + non-zero-normalized continuous", y=1.01)
plt.tight_layout()
plt.show()

# Summary: what the new parameter_columns would look like
new_cols = [
    "m_norm", "sigma_norm", "epsilon_k_norm",
    "mu_bool", "mu_val",
    "kappa_ab_bool", "kappa_ab_val",
    "epsilon_k_ab_bool", "epsilon_k_ab_val",
]
print(f"Current parameter_columns ({6} params, input dim {6+2}):")
print(["m_norm", "sigma_norm", "epsilon_k_norm", "mu_norm", "kappa_ab_norm", "epsilon_k_ab_norm"])
print(f"\nProposed parameter_columns ({len(new_cols)} params, input dim {len(new_cols)+2}):")
print(new_cols)

In [ ]:
# Final parameter columns per README: 8 params, input dim 10
# is_associating / has_dipole: physics-motivated flags (kappa_ab and epsilon_k_ab are always zero together)
# kappa_ab_val, mu_val: log-normalized over non-zero range (spans orders of magnitude)
# epsilon_k_ab_val: linear-normalized over non-zero range (already well-distributed)

bimodal_raw = ["mu", "kappa_ab", "epsilon_k_ab"]
log_cols    = ["mu", "kappa_ab"]   # log-transform before normalizing
linear_cols = ["epsilon_k_ab"]     # linear normalize

saft["is_associating"] = (saft["kappa_ab"] > 0).astype(float)
saft["has_dipole"]     = (saft["mu"]       > 0).astype(float)

for rc in bimodal_raw:
    col     = saft[rc].values
    nz_mask = col > 0
    nz_vals = col[nz_mask]
    val     = np.zeros(len(col), dtype=float)
    if len(nz_vals) > 0:
        if rc in log_cols:
            nz_transformed = np.log(nz_vals)
        else:
            nz_transformed = nz_vals
        v_min, v_max = nz_transformed.min(), nz_transformed.max()
        val[nz_mask] = (nz_transformed - v_min) / (v_max - v_min + 1e-12)
        print( rc, v_min, v_max )
    saft[f"{rc}_val"] = val

# Drop the stale _bool columns added in full_data_4
for col in ["epsilon_k_ab_bool", "kappa_ab_bool", "mu_bool"]:
    if col in saft.columns:
        saft.drop(columns=col, inplace=True)

parameter_columns = [
    "m_norm", "sigma_norm", "epsilon_k_norm",
    "is_associating", "kappa_ab_val", "epsilon_k_ab_val",
    "has_dipole", "mu_val",
]

out_path = path
saft.to_csv(out_path, index=False)
print(f"Saved {len(saft)} rows to {out_path}")
print(f"parameter_columns ({len(parameter_columns)} params, input dim {len(parameter_columns)+2}):")
print(parameter_columns)
saft[parameter_columns].describe().round(4)
